# 🏥 ISIC MILK10k Benchmark — Training Notebook

**Task:** Multi-category skin lesion diagnosis (11 classes)  
**Metric:** Macro F1 (threshold = 0.5)  
**Challenge:** https://challenge.isic-archive.com/landing/milk10k/

---
### Trước khi chạy:
1. `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (cell bên dưới)
3. Upload dataset vào Drive theo đúng cấu trúc thư mục

### Cấu trúc thư mục trên Drive:
```
MyDrive/MILK10k/
└── MILK10K_SOLUTION/
    ├── datasets/MILK10k/
    │   ├── train/
    │   │   ├── MILK10k_Training_Metadata.csv
    │   │   ├── MILK10k_Training_GroundTruth.csv
    │   │   └── MILK10k_Training_Input/<lesion_id>/<isic_id>.jpg
    │   └── test/
    │       ├── MILK10k_Test_Metadata.csv
    │       └── MILK10k_Test_Input/<lesion_id>/<isic_id>.jpg
    ├── models/
    ├── src/
    ├── configs/
    └── train.py
```

## 🔧 Bước 1 — Kiểm tra GPU & Mount Drive

In [ ]:
# Kiểm tra GPU
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Không tìm thấy GPU — hãy đổi runtime type!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
import os

# ════════════════════════════════════════════════════════
# CẤU HÌNH ĐƯỜNG DẪN — chỉnh sửa nếu cần
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/MILK10k/MILK10K_SOLUTION"
# ════════════════════════════════════════════════════════

os.chdir(DRIVE_PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

## 📦 Bước 2 — Cài đặt thư viện

In [ ]:
%%capture install_output
!pip install timm>=0.9.12 albumentations>=1.3.1 iterative-stratification tqdm
print("Installation done ✓")

In [ ]:
# Verify imports
import timm, albumentations, sklearn
print(f"timm          : {timm.__version__}")
print(f"albumentations: {albumentations.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print("All libraries OK ✓")

## 📂 Bước 3 — Chuẩn bị data (chỉ chạy 1 lần)

In [ ]:
import os
from pathlib import Path

COMBINED_TRAIN = Path("datasets/MILK10k/train/train_combined.csv")
COMBINED_TEST  = Path("datasets/MILK10k/test/test_combined.csv")

if COMBINED_TRAIN.exists() and COMBINED_TEST.exists():
    print("✓ train_combined.csv và test_combined.csv đã có sẵn — bỏ qua bước này")
else:
    print("Đang tạo combined CSVs...")
    !python src/prepare_data.py --base_dir datasets/MILK10k
    print("✓ Xong!")

In [ ]:
SPLIT_TRAIN = Path("datasets/MILK10k/splits/train_fold0.csv")

if SPLIT_TRAIN.exists():
    print("✓ Splits đã có sẵn — bỏ qua")
else:
    print("Đang tạo train/val splits (5-fold)...")
    for fold in range(5):
        print(f"  Fold {fold}...")
        os.system(
            f"python src/split_data.py "
            f"--csv datasets/MILK10k/train/train_combined.csv "
            f"--out_dir datasets/MILK10k/splits "
            f"--fold {fold} --n_folds 5 --seed 42"
        )
    print("✓ Xong!")

In [ ]:
# Kiểm tra data
import pandas as pd

train_df = pd.read_csv("datasets/MILK10k/splits/train_fold0.csv")
val_df   = pd.read_csv("datasets/MILK10k/splits/val_fold0.csv")
test_df  = pd.read_csv("datasets/MILK10k/test/test_combined.csv")

print(f"Train fold0 : {len(train_df):,} lesions")
print(f"Val fold0   : {len(val_df):,} lesions")
print(f"Test        : {len(test_df):,} lesions")

LABEL_COLS = ["AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC"]
print("\nClass distribution (train fold0):")
for c in LABEL_COLS:
    n = int(train_df[c].sum())
    print(f"  {c:<10}: {n:>5} ({100*n/len(train_df):>5.1f}%)")

## ⚙️ Bước 4 — Cấu hình training

Chọn model và điều chỉnh hyperparameters tại đây:

In [ ]:
# ════════════════════════════════════════════════════════
# CHỌN CONFIG — đổi tên file để chọn model
# Options: swin_base | convnext_base | efficientnet_b3 | maxvit_tiny | vit_base
CONFIG = "configs/swin_base.yaml"
# ════════════════════════════════════════════════════════

import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

# Override để tối ưu cho Colab T4 (15 GB VRAM)
OVERRIDES = {
    "epochs"      : 30,
    "batch_size"  : 32,    # giảm xuống 16 nếu OOM
    "num_workers" : 2,     # Colab thường dùng 2
    "image_size"  : 224,
    "lr"          : 1e-4,
    "loss_name"   : "bce",
    "use_pos_weight" : True,
    "use_amp"     : True,
    "early_stopping_patience": 8,
}
cfg.update(OVERRIDES)

print(f"Model   : {cfg['model_name']}")
print(f"Mode    : {cfg.get('mode','single_image')}")
print(f"Epochs  : {cfg['epochs']}")
print(f"Batch   : {cfg['batch_size']}")
print(f"LR      : {cfg['lr']}")
print(f"Loss    : {cfg['loss_name']}")
print(f"ImgSize : {cfg['image_size']}")

## 🚀 Bước 5 — Train model

In [ ]:
import sys
sys.path.insert(0, ".")

from src.dataset import build_meta_processor
from models.model_factory import build_model
from src.train import train
from src.utils import set_seed

set_seed(cfg.get("seed", 42))

# Build metadata processor nếu dùng metadata
meta_dim = 0
if cfg.get("use_metadata", False):
    proc     = build_meta_processor(pd.read_csv(cfg["train_csv"]))
    meta_dim = proc.meta_dim
    print(f"Metadata dim: {meta_dim}")

# Build model
model = build_model(cfg, meta_dim=meta_dim)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {cfg['model_name']}")
print(f"Params: {n_params:,}")

In [ ]:
# Training
best_f1 = train(cfg, model)
print(f"\n✅ Training complete! Best val macro F1 = {best_f1:.4f}")

## 📊 Bước 6 — Xem training log

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

model_name = cfg["model_name"]
log_path   = f"outputs/logs/{model_name}_train_log.csv"

log_df = pd.read_csv(log_path)
print(log_df.tail(10).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log_df["epoch"], log_df["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(log_df["epoch"], log_df["val_loss"],   label="Val Loss",   color="tomato")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(log_df["epoch"], log_df["macro_f1"], color="green", marker="o", markersize=3)
axes[1].set_title("Val Macro F1"); axes[1].set_xlabel("Epoch")
axes[1].axhline(log_df["macro_f1"].max(), color="gray", linestyle="--",
                label=f'Best={log_df["macro_f1"].max():.4f}')
axes[1].legend()

f1_cols = [c for c in log_df.columns if c.startswith("f1_")]
best_row = log_df.loc[log_df["macro_f1"].idxmax()]
class_names = [c.replace("f1_", "") for c in f1_cols]
class_f1s   = [best_row[c] for c in f1_cols]
axes[2].barh(class_names, class_f1s, color="steelblue")
axes[2].set_title(f"Per-class F1 @ best epoch ({int(best_row['epoch'])})")
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f"outputs/logs/{model_name}_training_curves.png", dpi=150)
plt.show()
print(f"Best macro F1: {log_df['macro_f1'].max():.4f} at epoch {log_df['macro_f1'].idxmax()+1}")

## 🔍 Bước 7 — Inference & tạo submission

In [ ]:
from src.infer import infer

model_name   = cfg["model_name"]
checkpoint   = f"outputs/checkpoints/{model_name}/best.pth"
test_csv     = "datasets/MILK10k/test/test_combined.csv"
test_img_dir = "datasets/MILK10k/test/MILK10k_Test_Input"
out_path     = f"outputs/submissions/submission_{model_name}.csv"

print(f"Checkpoint : {checkpoint}")
print(f"Test CSV   : {test_csv}")
print(f"Output     : {out_path}")

infer(
    config_path = CONFIG,
    checkpoint  = checkpoint,
    test_csv    = test_csv,
    out_path    = out_path,
    image_dir   = test_img_dir,
    use_tta     = False,   # đổi True để bật TTA
)

In [ ]:
# Kiểm tra submission
import pandas as pd

sub = pd.read_csv(out_path)
print(f"Submission rows    : {len(sub)}")
print(f"Columns            : {sub.columns.tolist()}")
print(f"NaN count          : {sub.isnull().sum().sum()}")
print(f"Values in [0,1]?   : {((sub.iloc[:,1:] >= 0) & (sub.iloc[:,1:] <= 1)).all().all()}")
print("\nFirst 5 rows:")
display(sub.head())
print("\nProbability statistics:")
display(sub.iloc[:,1:].describe().round(3))

## 🔄 Bước 8 — Train với TTA inference (optional)

Bật TTA để cải thiện ~1-2% F1:

In [ ]:
out_tta = f"outputs/submissions/submission_{model_name}_tta.csv"

infer(
    config_path = CONFIG,
    checkpoint  = checkpoint,
    test_csv    = test_csv,
    out_path    = out_tta,
    image_dir   = test_img_dir,
    use_tta     = True,
)
print(f"TTA submission saved → {out_tta}")

## 🔁 Bước 9 — Train tất cả 5 folds (Cross-Validation)

Train 5 folds và ensemble để có kết quả tốt nhất:

In [ ]:
import yaml, sys
sys.path.insert(0, ".")
from src.dataset import build_meta_processor
from models.model_factory import build_model
from src.train import train
from src.infer import infer
from src.utils import set_seed
import pandas as pd

with open(CONFIG) as f:
    base_cfg = yaml.safe_load(f)
base_cfg.update(OVERRIDES)

fold_f1s    = []
fold_subs   = []

for fold in range(5):
    print(f"\n{'='*50}")
    print(f" FOLD {fold}/4")
    print(f"{'='*50}")

    cfg_fold = base_cfg.copy()
    cfg_fold["train_csv"] = f"datasets/MILK10k/splits/train_fold{fold}.csv"
    cfg_fold["val_csv"]   = f"datasets/MILK10k/splits/val_fold{fold}.csv"
    cfg_fold["checkpoint_dir"] = f"outputs/checkpoints/fold{fold}"

    set_seed(42 + fold)

    meta_dim = 0
    if cfg_fold.get("use_metadata", False):
        proc     = build_meta_processor(pd.read_csv(cfg_fold["train_csv"]))
        meta_dim = proc.meta_dim

    model  = build_model(cfg_fold, meta_dim=meta_dim)
    best_f1 = train(cfg_fold, model)
    fold_f1s.append(best_f1)

    # Inference cho fold này
    ckpt     = f"outputs/checkpoints/fold{fold}/{cfg_fold['model_name']}/best.pth"
    sub_path = f"outputs/submissions/submission_fold{fold}.csv"
    infer(
        config_path=CONFIG, checkpoint=ckpt,
        test_csv=test_csv, out_path=sub_path,
        image_dir=test_img_dir, use_tta=True,
    )
    fold_subs.append(sub_path)
    print(f"Fold {fold} best F1: {best_f1:.4f}")

print(f"\n{'='*50}")
print(f"CV Results: {[f'{f:.4f}' for f in fold_f1s]}")
print(f"Mean F1   : {sum(fold_f1s)/len(fold_f1s):.4f}")

In [ ]:
# Ensemble tất cả 5 folds
from src.submission import ensemble_submissions

ensemble_submissions(
    paths    = fold_subs,
    weights  = None,   # uniform average
    out_path = "outputs/submissions/submission_5fold_ensemble.csv",
)
print("✅ 5-fold ensemble submission saved!")

## 🧪 Bước 10 — Ensemble nhiều model

Để kết quả tốt nhất, train Swin + ConvNeXt + EfficientNet rồi ensemble:

In [ ]:
# Ví dụ ensemble 3 model submissions
from src.submission import ensemble_submissions
from pathlib import Path

submissions_to_ensemble = [
    "outputs/submissions/submission_swin_base_patch4_window7_224_tta.csv",
    "outputs/submissions/submission_convnext_base_tta.csv",
    "outputs/submissions/submission_efficientnet_b3_tta.csv",
]

# Chỉ ensemble những file đã tồn tại
existing = [p for p in submissions_to_ensemble if Path(p).exists()]
print(f"Found {len(existing)}/{len(submissions_to_ensemble)} submissions to ensemble")

if len(existing) >= 2:
    ensemble_submissions(
        paths    = existing,
        weights  = None,
        out_path = "outputs/submissions/submission_multi_model_ensemble.csv",
    )
else:
    print("Cần ít nhất 2 submission để ensemble")

## 💾 Bước 11 — Download submission

In [ ]:
from google.colab import files
import os

# Hiển thị tất cả submissions
subs = list(Path("outputs/submissions").glob("*.csv"))
print("Submissions available:")
for s in sorted(subs):
    size = s.stat().st_size / 1024
    print(f"  {s.name:60s} {size:>6.1f} KB")

# Download file tốt nhất
# Chọn file nào muốn download:
best_sub = "outputs/submissions/submission_swin_base_patch4_window7_224_tta.csv"
if os.path.exists(best_sub):
    files.download(best_sub)
    print(f"Downloading {best_sub}")
else:
    print("File không tồn tại, chọn file khác:")
    if subs:
        files.download(str(subs[-1]))

---
## 📝 Quick Reference

| Lệnh | Mô tả |
|------|-------|
| `python src/prepare_data.py` | Tạo combined CSV từ raw data |
| `python src/split_data.py --fold 0` | Tạo train/val split |
| `python train.py --config configs/swin_base.yaml` | Train Swin |
| `python train.py --config configs/convnext_base.yaml` | Train ConvNeXt |
| `python src/infer.py --tta ...` | Inference với TTA |

### Recommended training order:
1. **convnext_base** (ASL loss) → thường cho F1 cao nhất
2. **swin_base** (BCE + pos_weight)
3. **efficientnet_b3** (Focal loss)
4. Ensemble 3 model

### Tips cho Colab:
- Dùng **T4 GPU** (miễn phí) hoặc **A100** (Colab Pro)
- Lưu checkpoint vào Drive ngay (`checkpoint_dir` trỏ vào Drive)
- `batch_size=16` nếu OOM với T4
- `num_workers=2` là tối ưu cho Colab
- Bật `use_amp=true` để tiết kiệm VRAM